# Use Case 1 — Benchmark Qwen3-Coder-30B across G5, G6, and G7

This notebook accompanies the AWS blog post *Benchmarking Small LLM Inference on SageMaker AI: G7 vs G5 and G6*.
It benchmarks **`Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8`** — a 30B Mixture-of-Experts coding model — across three
GPU instance families on **Amazon SageMaker AI real-time inference**, using **SageMaker AI generative AI benchmarking**:

| | |
|---|---|
| **Model** | Qwen3-Coder-30B-A3B-Instruct-FP8 |
| **Serving framework** | DJL Large Model Inference (LMI) 28.0 |
| **Instances** | `ml.g5.12xlarge` (A10G) · `ml.g6.12xlarge` (L4) · `ml.g7.12xlarge` (RTX PRO 4500 Blackwell) |
| **Benchmark tool** | SageMaker AI generative AI benchmarking (NVIDIA AIPerf) |
| **Workload** | 128 input tokens · 128 output tokens · concurrency 4 · 100 requests |

We use the **same model, serving container, and workload** for every instance so the GPU generation is the
primary variable — an apples-to-apples comparison. For each instance we deploy **one** endpoint and run **two**
benchmark passes against it:

1. **Non-streaming** — output-token throughput, request throughput, and request latency (avg / P50 / P90 / P99).
2. **Streaming** — time to first token (TTFT) and inter-token latency (ITL) for interactive applications.

The workflow is **Configure → Deploy + Benchmark (looped over all three instances) → Compare**. The notebook is
self-contained: run it top to bottom to reproduce the results. Endpoints are always torn down after each instance.

### Prerequisites
- An AWS account with Amazon SageMaker AI access, run from **SageMaker Studio** (or set `SAGEMAKER_ROLE_ARN`).
- An IAM execution role with `AmazonSageMakerFullAccess` and S3 read/write on your SageMaker default bucket.
- **Instance quota** for `ml.g5.12xlarge`, `ml.g6.12xlarge`, and `ml.g7.12xlarge` endpoint usage in your Region.
- Python 3.10+ with the SageMaker Python SDK (`sagemaker>=3.16.0`).

> **Note on the container:** `image_uris.retrieve('djl-lmi', 'latest')` currently resolves to LMI 27, which
> **cannot** run on G7 (Blackwell / `sm_120`) — it crash-loops with `cudaErrorUnsupportedPtxVersion`. **LMI 28**
> adds `sm_120` support *and* runs on G5/G6, so we pin one image across all three instances.

## 1. Setup

In [ ]:
%pip install -q "sagemaker>=3.16.0"

In [ ]:
import os, json, time, uuid, logging, sys, traceback
from datetime import datetime

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("uc1")

sess = Session()
REGION = sess.boto_session.region_name
account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
# In Studio the role auto-resolves; elsewhere set SAGEMAKER_ROLE_ARN.
ROLE = os.environ.get("SAGEMAKER_ROLE_ARN") or get_execution_role(sagemaker_session=sess)
sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

print("Region      :", REGION)
print("Account     :", account_id)
print("Role        :", ROLE)

## 2. Configure the benchmark

Edit `INSTANCES`, the serving knobs, and the workload here. Everything downstream is derived from this cell.

In [ ]:
# ---- Model ----
HF_MODEL_ID  = "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8"
MODEL_TAG    = "qwen3-coder-30b-a3b-fp8"
MODEL_S3_URI = f"s3://sagemaker-{REGION}-{account_id}/lmi-models/{MODEL_TAG}/"

# ---- Instances to benchmark (looped in order) ----
INSTANCES = ["ml.g5.12xlarge", "ml.g6.12xlarge", "ml.g7.12xlarge"]

# ---- Serving config (identical across instances for a fair comparison) ----
TENSOR_PARALLEL_DEGREE = 2      # REQUIRED: Qwen FP8 is block-quantized (block_n=128); TP=4 fails divisibility.
MAX_MODEL_LEN          = 4096
MAX_ROLLING_BATCH_SIZE = 16

# ---- LMI / DJL container (pin LMI 28 -> Blackwell/sm_120 support for G7 + runs on G5/G6) ----
LMI_IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:0.36.0-lmi28.0.0-cu130"

# ---- Benchmark workload (identical across instances) ----
BENCH = dict(concurrency=4, request_count=100, prompt_input_tokens_mean=128, output_tokens_mean=128)

# ---- (Optional) SageMaker Hosting on-demand $/hr, for the cost-per-token view in section 7.
# Verify against the current pricing page for your Region: https://aws.amazon.com/sagemaker/pricing/
PRICE_PER_HR = {
    "ml.g5.12xlarge": 7.09,
    "ml.g6.12xlarge": 5.79,
    "ml.g7.12xlarge": 8.91,
}

# ---- Timeouts + where to write results ----
HEALTH_CHECK_TIMEOUT = 3600     # container startup health-check budget (30B FP8 model)
DEPLOY_POLL_TIMEOUT  = 2700     # wait-for-InService budget
BENCH_POLL_TIMEOUT   = 2400     # per benchmark-job budget
RESULTS_DIR = "results/uc1"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Instances   :", INSTANCES)
print("Container   :", LMI_IMAGE)
print("Model S3 URI:", MODEL_S3_URI)
print("Serving     : vLLM | TP =", TENSOR_PARALLEL_DEGREE, "| max_model_len =", MAX_MODEL_LEN,
      "| max_rolling_batch_size =", MAX_ROLLING_BATCH_SIZE)
print("Workload    :", BENCH)

## 3. Stage the model artifacts

### 3a. Download the model from Hugging Face to S3

The weights (~29 GiB) are uploaded once and reused on every run. If they are already in S3, this step is skipped.

In [ ]:
_bucket = f"sagemaker-{REGION}-{account_id}"
_prefix = f"lmi-models/{MODEL_TAG}"
try:
    _already = s3.list_objects_v2(Bucket=_bucket, Prefix=f"{_prefix}/").get("KeyCount", 0) > 0
except Exception:
    _already = False
print("Model already in S3 — skipping download." if _already else "Model not in S3 — will download from Hugging Face.")

In [ ]:
if _already:
    print("Using existing", MODEL_S3_URI)
else:
    import subprocess, pathlib, shutil
    HF_TOKEN  = None                     # set to "hf_..." only if the repo is gated (this one is public)
    LOCAL_DIR = pathlib.Path("./downloaded_model")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[hf_transfer]"])
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

    from huggingface_hub import snapshot_download
    print(f"Downloading {HF_MODEL_ID} -> {LOCAL_DIR} ...")
    snapshot_download(repo_id=HF_MODEL_ID, local_dir=str(LOCAL_DIR), token=HF_TOKEN,
                      ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*", "rust_model*", "*.ot", "original/"])

    # ensure the bucket exists
    try:
        s3.head_bucket(Bucket=_bucket)
    except s3.exceptions.ClientError:
        if REGION == "us-east-1":
            s3.create_bucket(Bucket=_bucket)
        else:
            s3.create_bucket(Bucket=_bucket, CreateBucketConfiguration={"LocationConstraint": REGION})
        print("Created bucket:", _bucket)

    files = [f for f in LOCAL_DIR.rglob("*") if f.is_file()]
    print(f"Uploading {len(files)} files to s3://{_bucket}/{_prefix}/ ...")
    for i, f in enumerate(files, 1):
        s3.upload_file(str(f), _bucket, f"{_prefix}/{f.relative_to(LOCAL_DIR)}")
        if i % 10 == 0 or i == len(files):
            print(f"  {i}/{len(files)} uploaded")
    shutil.rmtree(LOCAL_DIR)
    print("Done. MODEL_S3_URI =", MODEL_S3_URI)

### 3b. Package the serving configuration

We use **LMI's native OpenAI-compatible serving** — a `serving.properties` file with no custom handler. That one endpoint answers both streaming and non-streaming requests, which lets us drive it with `Workload.synthetic(...)` in both passes.

In [ ]:
serving_properties = f'''option.model_id={MODEL_S3_URI}
option.rolling_batch=vllm
option.tensor_parallel_degree={TENSOR_PARALLEL_DEGREE}
option.max_rolling_batch_size={MAX_ROLLING_BATCH_SIZE}
option.dtype=auto
option.max_model_len={MAX_MODEL_LEN}
option.trust_remote_code=true
'''

os.makedirs("model_artifacts", exist_ok=True)
with open("model_artifacts/serving.properties", "w") as f:
    f.write(serving_properties)
print(serving_properties)

art_prefix = f"lmi-models/{MODEL_TAG}-uc1-artifacts/" + datetime.now().strftime("%Y%m%d%H%M%S")
s3.upload_file("model_artifacts/serving.properties", _bucket, f"{art_prefix}/serving.properties")
S3_ARTIFACT_URI = f"s3://{_bucket}/{art_prefix}/"
print("Artifacts uploaded to:", S3_ARTIFACT_URI)

## 4. Deploy and benchmark each instance

For every instance we: **deploy** the endpoint → **smoke test** it → run the **non-streaming** benchmark →
run the **streaming** benchmark → **tear everything down**. Cleanup runs in a `finally` block, so a failure on
one instance never leaks resources or blocks the others.

We poll with plain `boto3` calls and simple prints (no progress widgets) so the notebook executes cleanly
under automation such as `papermill`.

In [ ]:
from sagemaker.core.resources import Model, EndpointConfig, Endpoint, AIWorkloadConfig
from sagemaker.serve import start_benchmark, Workload

TERMINAL = {"Completed", "Failed", "Stopped"}

def _metric(metrics, key):
    # Pull one AIPerf metric (avg/p50/p90/p99/...) into a plain dict.
    try:
        obj = metrics.get(key) if hasattr(metrics, "get") else getattr(metrics, key, None)
    except Exception:
        obj = None
    if obj is None:
        return None
    d = {}
    for stat in ("avg", "p50", "p90", "p99", "min", "max", "unit"):
        v = getattr(obj, stat, None)
        if v is not None:
            d[stat] = v
    return d or None

def _run_benchmark(endpoint, streaming, keys):
    # Start one benchmark pass, poll to completion, return {metric: {...}}.
    wl = Workload.synthetic(tokenizer=HF_MODEL_ID, streaming=streaming, **BENCH)
    job = start_benchmark(endpoint=endpoint, workload=wl, role=ROLE, wait=False)
    label = "streaming" if streaming else "non-streaming"
    print(f"    {label} benchmark job: {job.ai_benchmark_job_name}", flush=True)
    t0, status = time.time(), None
    deadline = time.time() + BENCH_POLL_TIMEOUT
    while time.time() < deadline:
        try:
            job.refresh(); status = str(job.ai_benchmark_job_status)
        except Exception as e:
            status = f"poll_error: {e}"
        print(f"      [{int(time.time()-t0)}s] {label}: {status}", flush=True)
        if status in TERMINAL:
            break
        time.sleep(20)
    out = {"status": status}
    if status == "Completed":
        M = job.show_result().metrics
        for k in keys:
            v = _metric(M, k)
            if v:
                out[k] = v
    else:
        out["failure_reason"] = getattr(job, "failure_reason", None)
    # tear down the benchmark job + its workload config
    for lbl, fn in [("benchmark job", job.delete),
                    ("workload config",
                     lambda: AIWorkloadConfig.get(ai_workload_config_name=job.ai_workload_config_identifier).delete())]:
        try:
            fn()
        except Exception as e:
            print(f"      [skip] {lbl}: {e}")
    return out

NONSTREAM_KEYS = ["request_throughput", "output_token_throughput", "request_latency"]
STREAM_KEYS    = ["time_to_first_token", "inter_token_latency", "output_token_throughput", "request_throughput"]

def run_instance(instance_type):
    short = instance_type.replace("ml.", "").replace(".", "-")
    uid   = f"{int(time.time())}-{uuid.uuid4().hex[:6]}"
    names = dict(model=f"qwen-uc1-{short}-{uid}",
                 epc=f"qwen-uc1-{short}-epc-{uid}",
                 ep=f"qwen-uc1-{short}-ep-{uid}")
    rec = {"instance_type": instance_type, "container": LMI_IMAGE.split("/")[-1], "status": "started"}
    endpoint = None
    print(f"\n{'='*70}\n{instance_type}\n{'='*70}", flush=True)
    try:
        log.info("[%s] creating model + endpoint", instance_type)
        Model.create(model_name=names["model"], execution_role_arn=ROLE,
            primary_container={"image": LMI_IMAGE, "model_data_source": {"s3_data_source": {
                "s3_uri": S3_ARTIFACT_URI, "s3_data_type": "S3Prefix", "compression_type": "None"}}})
        EndpointConfig.create(endpoint_config_name=names["epc"], production_variants=[{
            "variant_name": "AllTraffic", "model_name": names["model"], "instance_type": instance_type,
            "initial_instance_count": 1,
            "container_startup_health_check_timeout_in_seconds": HEALTH_CHECK_TIMEOUT}])
        t0 = time.time()
        endpoint = Endpoint.create(endpoint_name=names["ep"], endpoint_config_name=names["epc"])
        log.info("[%s] waiting for InService (10-20 min for a 30B FP8 model)...", instance_type)
        status = None; deadline = time.time() + DEPLOY_POLL_TIMEOUT
        while time.time() < deadline:
            d = sm.describe_endpoint(EndpointName=names["ep"]); status = d["EndpointStatus"]
            print(f"    [{int(time.time()-t0)}s] {status}", flush=True)
            if status == "InService":
                break
            if status == "Failed":
                raise RuntimeError(f"Endpoint failed: {d.get('FailureReason')}")
            time.sleep(20)
        if status != "InService":
            raise RuntimeError(f"not InService after {DEPLOY_POLL_TIMEOUT}s (last {status})")
        rec["deploy_seconds"] = round(time.time() - t0, 1)
        print(f"    InService in {rec['deploy_seconds']}s", flush=True)

        # smoke test (OpenAI chat schema)
        payload = {"messages": [{"role": "user", "content": "Write a Python function for the nth Fibonacci number."}],
                   "max_tokens": 64, "temperature": 0.2}
        resp = endpoint.invoke(body=json.dumps(payload), content_type="application/json", accept="application/json")
        out = json.loads(resp.body.read().decode("utf-8"))
        rec["smoke_sample"] = (out["choices"][0]["message"]["content"] if isinstance(out, dict) and out.get("choices")
                               else str(out))[:200]
        print("    smoke OK:", rec["smoke_sample"][:120], flush=True)

        # two benchmark passes against the same endpoint
        print("    --- non-streaming pass ---", flush=True)
        rec["nonstreaming"] = _run_benchmark(endpoint, streaming=False, keys=NONSTREAM_KEYS)
        print("    --- streaming pass ---", flush=True)
        rec["streaming"] = _run_benchmark(endpoint, streaming=True, keys=STREAM_KEYS)

        ok = (rec["nonstreaming"].get("status") == "Completed" and rec["streaming"].get("status") == "Completed")
        rec["status"] = "success" if ok else "partial"
    except Exception as e:
        rec["status"] = "failed"; rec["error"] = str(e)[:400]
        log.error("[%s] FAILED: %s", instance_type, traceback.format_exc())
    finally:
        def _try(lbl, fn):
            try:
                fn(); print("    deleted:", lbl)
            except Exception as e:
                print("    [skip]", lbl, e)
        _try("endpoint", lambda: sm.delete_endpoint(EndpointName=names["ep"]))
        _try("endpoint config", lambda: sm.delete_endpoint_config(EndpointConfigName=names["epc"]))
        _try("model", lambda: sm.delete_model(ModelName=names["model"]))
        print(f"    [{instance_type}] cleaned up (status={rec['status']})", flush=True)
    return rec

Run the loop. This deploys, benchmarks, and tears down each instance in turn — budget roughly **60–90 minutes** for all three.

In [ ]:
results = []
for inst in INSTANCES:
    rec = run_instance(inst)
    results.append(rec)
    with open(os.path.join(RESULTS_DIR, f"qwen-uc1-{inst.replace('ml.','').replace('.','-')}.json"), "w") as f:
        json.dump(rec, f, indent=2, default=str)
print("\nAll instances done:", [(r["instance_type"], r["status"]) for r in results])

## 5. Non-streaming results

Output-token throughput, request throughput, and request latency (avg / P50 / P90 / P99) — the primary comparison.

In [ ]:
import pandas as pd
from IPython.display import display

def g(rec, section, metric, stat):
    try:
        return rec.get(section, {}).get(metric, {}).get(stat)
    except Exception:
        return None

rows = []
for r in results:
    ns = r.get("nonstreaming", {})
    rows.append({
        "instance": r["instance_type"],
        "status": r["status"],
        "output tok/s": g(r, "nonstreaming", "output_token_throughput", "avg"),
        "req/s": g(r, "nonstreaming", "request_throughput", "avg"),
        "latency avg (ms)": g(r, "nonstreaming", "request_latency", "avg"),
        "P50 (ms)": g(r, "nonstreaming", "request_latency", "p50"),
        "P90 (ms)": g(r, "nonstreaming", "request_latency", "p90"),
        "P99 (ms)": g(r, "nonstreaming", "request_latency", "p99"),
        "deploy (s)": r.get("deploy_seconds"),
    })
ns_df = pd.DataFrame(rows).set_index("instance")
display(ns_df.round(1))

## 6. Streaming results — TTFT and inter-token latency

For interactive coding applications, `streaming=True` surfaces **time to first token** (how fast output starts) and **inter-token latency** (how smoothly it continues).

In [ ]:
rows = []
for r in results:
    rows.append({
        "instance": r["instance_type"],
        "TTFT avg (ms)": g(r, "streaming", "time_to_first_token", "avg"),
        "TTFT P50 (ms)": g(r, "streaming", "time_to_first_token", "p50"),
        "TTFT P99 (ms)": g(r, "streaming", "time_to_first_token", "p99"),
        "ITL avg (ms)":  g(r, "streaming", "inter_token_latency", "avg"),
        "ITL P50 (ms)":  g(r, "streaming", "inter_token_latency", "p50"),
        "ITL P99 (ms)":  g(r, "streaming", "inter_token_latency", "p99"),
        "output tok/s":  g(r, "streaming", "output_token_throughput", "avg"),
    })
stream_df = pd.DataFrame(rows).set_index("instance")
display(stream_df.round(1))

## 7. (Optional) Cost per 1M output tokens

A derived price-performance view from the non-streaming throughput:

$$\text{\$ per 1M output tokens} = \frac{\text{\$/hr} \times 1{,}000{,}000}{\text{output tokens/second} \times 3600}$$

Fill `PRICE_PER_HR` in the config cell with current SageMaker Hosting on-demand prices for your Region.

In [ ]:
rows = []
for r in results:
    inst = r["instance_type"]
    tok  = g(r, "nonstreaming", "output_token_throughput", "avg")
    hr   = PRICE_PER_HR.get(inst)
    permil = round(hr * 1e6 / (float(tok) * 3600), 2) if (hr and tok) else None
    rows.append({"instance": inst, "$/hr": hr, "output tok/s": tok, "$/1M output tok": permil})
cost_df = pd.DataFrame(rows).set_index("instance")
display(cost_df.round(2))
best = cost_df["$/1M output tok"].dropna()
if not best.empty:
    print("Lowest cost per token:", best.idxmin(), "at $%.2f / 1M" % best.min())

## 8. Save the comparison and verify cleanup

Write the tables to `results/uc1/` and confirm no benchmark endpoints remain.

In [ ]:
# save markdown + csv
with open(os.path.join(RESULTS_DIR, "comparison.md"), "w") as f:
    f.write("# UC1 — Qwen3-Coder-30B FP8: G5 vs G6 vs G7\n\n")
    f.write("## Non-streaming\n\n" + ns_df.round(1).to_markdown() + "\n\n")
    f.write("## Streaming (TTFT / ITL)\n\n" + stream_df.round(1).to_markdown() + "\n\n")
    f.write("## Cost per 1M output tokens\n\n" + cost_df.round(2).to_markdown() + "\n")
ns_df.to_csv(os.path.join(RESULTS_DIR, "nonstreaming.csv"))
stream_df.to_csv(os.path.join(RESULTS_DIR, "streaming.csv"))
print("Saved comparison to", RESULTS_DIR)

# verify no leftover UC1 endpoints
left = [e["EndpointName"] for e in sm.list_endpoints(NameContains="qwen-uc1").get("Endpoints", [])]
print("Remaining UC1 endpoints:", left if left else "none")